In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from torch.distributions import Categorical


In [2]:
GRID_SIZE = 5
ACTION = [
    (-1,0), (0,1),
    (1,0), (0,-1),
    (0,0)
    ]
ACTION_SYMBOLS = [
    '↑', '→',
    '↓', '←',
    'O'
    ]
GOAL_STATE = (3, 2)
OBSTACLE = [
    (1,1), (1,2),
    (2,2),
    (3,1), (3,3),
    (4,1)
]


GAMMA = 0.9
ALPHA = 0.001
num_episodes = 2000
num_steps = 1000
state_dim = 25
action_dim = len(ACTION)
obstacle_through = True

In [3]:
"""
提取訓練好的 DQN 神經網絡中的 Q 值，還原成一個 (GRID_SIZE, GRID_SIZE, action_dim) 的 Q 表
"""
def extract_q_table_from_net(trained_net, env, grid_size, action_dim):
    """
    将训练好的 DQN 神经网络，还原成 (GRID_SIZE, GRID_SIZE, action_dim) 的 3D Q 表
    """
    # 1. 准备一个全 0 的空白 Q 表
    Q_table = np.zeros((grid_size, grid_size, action_dim))
    
    # 2. 将网络切换到评估模式 (锁定参数)
    trained_net.eval() 
    
    # 3. 暴力遍历地图上的每一个坐标
    for r in range(grid_size):
        for c in range(grid_size):
            state = (r, c)
            
            state_vec = env._get_state_vector(state)
            
            # 转换成 PyTorch 需要的 Tensor 格式 (别忘了升维变成 Batch Size = 1)
            state_tensor = torch.FloatTensor(state_vec).unsqueeze(0)
            
            # 让大脑进行推演，算出这个坐标下 5 个动作的 Q 值
            with torch.no_grad():
                # .squeeze() 是把 (1, 5) 降维回 (5,)
                # .numpy() 是把 PyTorch Tensor 变回 NumPy 数组
                q_values = trained_net(state_tensor).squeeze().numpy()
                
            # 把这 5 个打分填入我们准备好的 Q 表对应位置
            Q_table[r, c] = q_values
            
    return Q_table

"""
提取最优策略 (Policy Extraction)
"""
def extract_policy(GRID_SIZE, OBSTACLE, Q_final, obstacle_through=True):
    final_policy = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    final_V = np.zeros((GRID_SIZE, GRID_SIZE))
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            state = (i, j)
            
            if (i, j) in OBSTACLE and not obstacle_through:
                continue 
            
            # 提取最大 Q 值对应的动作 (argmax)
            final_policy[i, j] = np.argmax(Q_final[i, j])
            # 提取最大 Q 值作为该格子的状态价值 V(s) (max)
            final_V[i, j] = np.max(Q_final[i, j])
    
    return final_policy, final_V

In [4]:
"""
可视化 (Heatmap & Policy Arrows)
"""
def visualize_grid(V, policy, Epsilon, OBSTACLE, obstacle_through=False):
    plt.figure(figsize=(8, 6))
    
    V_plot = np.copy(V)
    # 只有在不允许穿透时，才把障碍物设为 NaN 挖空（显示为背景色）
    # 如果允许穿透，我们要保留它惨烈的负分，让热力图显示出来
    if not obstacle_through:
        for (r, c) in OBSTACLE:
            V_plot[r, c] = np.nan
        
    ax = sns.heatmap(V_plot, annot=True, fmt=".2f", cmap="YlGnBu", 
                     linewidths=.5, cbar_kws={'label': 'State Value $V(s)$'})
    
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            state = (i, j)
            
            if not obstacle_through and state in OBSTACLE:
                # 不可穿透的墙，画上灰色大方块
                ax.text(j + 0.5, i + 0.5, '██', ha='center', va='center', color='gray', fontsize=20)
            
            elif state == GOAL_STATE:
                # 终点
                symbol = ACTION_SYMBOLS[policy[i, j]]
                ax.text(j + 0.5, i + 0.8, f"GOAL({symbol})", ha='center', va='center', color='red', weight='bold')
            
            else:
                symbol = ACTION_SYMBOLS[policy[i, j]]
                
                # 如果允许穿透，给障碍物格子加上特殊的色和紫色箭头
                if obstacle_through and state in OBSTACLE:
                    ax.text(j + 0.5, i + 0.8, symbol, ha='center', va='center', color='purple', fontsize=18, weight='bold')
                    ax.text(j + 0.5, i + 0.5, '██', ha='center', va='center', color='red', alpha=0.3, fontsize=20)
                else:
                    # 正常平地
                    ax.text(j + 0.5, i + 0.8, symbol, ha='center', va='center', color='black', fontsize=16, weight='bold')
                
    plt.title(f"Control ($\gamma$={GAMMA}, $\epsilon$={Epsilon}) | Obstacle Through: {obstacle_through}")
    plt.show()


def plot_learning_curve(learning_curve, window_size=100):
    """
    繪製學習曲線，包含原始回報和滑動平均回報
    """
    plt.figure(figsize=(10, 6))
    
    # 1. 繪製原始數據 (淺藍色，透明度低，用於觀察波動範圍)
    plt.plot(learning_curve, color='blue', alpha=0.2, label='Raw Total Reward')
    
    # 2. 計算並繪製滑動平均線 (紅色，較粗，用於觀察收斂趨勢)
    # 只有當 episode 數量大於窗口大小時才計算
    if len(learning_curve) >= window_size:
        # 使用 np.convolve 快速計算滑動平均
        moving_avg = np.convolve(learning_curve, np.ones(window_size)/window_size, mode='valid')
        # x 軸需要對齊，因為 valid 模式會砍掉前面的 window_size - 1 個點
        x_axis = np.arange(window_size - 1, len(learning_curve))
        plt.plot(x_axis, moving_avg, color='red', linewidth=2.5, label=f'Moving Average')
        
    plt.title('Learning Curve', fontsize=16)
    plt.xlabel('Episode', fontsize=14)
    plt.ylabel('Total Reward per Episode', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

In [5]:
"""
环境
"""
class GridEnv:
    def __init__(self, GRID_SIZE, GOAL_STATE, OBSTACLE, ACTION, obstacle_through=False):
        self.GRID_SIZE = GRID_SIZE
        self.GOAL_STATE = GOAL_STATE
        self.OBSTACLE = OBSTACLE
        self.ACTION = ACTION
        self.obstacle_through = obstacle_through

        self.state_dim = GRID_SIZE * GRID_SIZE
        self.current_state = None
    
    def _get_state_vector(self, state):
        """
        内部辅助函数：将二维坐标 (r, c) 转换为长度为 25 的 One-hot 一维向量
        因为 PyTorch 的神经网络需要浮点型的数组作为输入
        """
        vec = np.zeros(self.state_dim)
        r, c = state
        idx = r * self.GRID_SIZE + c
        vec[idx] = 1.0
        return vec
    
    def reset(self):
        """
        episode 重置：随机生成一个合法的起点，并返回其状态向量
        """
        while True:
            r = np.random.randint(self.GRID_SIZE)
            c = np.random.randint(self.GRID_SIZE)
            state = (r, c)
            if state == self.GOAL_STATE:
                continue
            if not self.obstacle_through and state in self.OBSTACLE:
                continue
            break

        self.current_state = state
        return self._get_state_vector(self.current_state)
    
    def step(self, action_idx):
        """
        执行动作：返回 next_state(向量形式), reward, done(是否结束)
        """
        r, c = self.current_state
        dr, dc = self.ACTION[action_idx]
        next_r, next_c = r + dr, c + dc

        done = False

        # 边界检查
        if next_r < 0 or next_r >= self.GRID_SIZE or next_c < 0 or next_c >= self.GRID_SIZE:
            reward = -1.0
            next_state = self.current_state
        
        else:
            next_state = (next_r, next_c)
            if self.obstacle_through and next_state in self.OBSTACLE:
                reward = -10.0

            elif not self.obstacle_through and next_state in self.OBSTACLE:
                reward = -10.0
                next_state = self.current_state
            
            elif next_state == self.GOAL_STATE:
                reward = 1.0
                done = True  # 到达终点，episode 结束
            
            else:
                reward = -0.1 # 每一步的惩罚，鼓励智能体尽快找到目标
        
        self.current_state = next_state

        return self._get_state_vector(self.current_state), reward, done


In [6]:
"""
Policy network
"""
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)
    

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        # 最后一层套上 Softmax，把它变成真正的概率分布 pi(a|s, theta)
        return torch.softmax(self.fc3(x), dim=1) # dim=1 表示对每一行进行 softmax, 输出的形状仍然是 (batch_size, action_dim)
    

In [ ]:
"""
REINFORCE 算法
更新公式: theta_{t+1} = theta_t + alpha * \Delta_theta(ln pi(a_t|s_t, theta_t)*q_t(s_t, a_t))
"""
def train_reinforce(env, num_episodes, num_steps, state_dim, action_dim, GAMMA, ALPHA, obstacle_through=False):

    env.obstacle_through = obstacle_through
    # 初始化策略网络和优化器
    policy_net = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy_net.parameters(), lr=ALPHA)

    return_history = []

    for ep in range(num_episodes):
        state = env.reset()

        # 用来存储这一个 episode 中的状态、动作和奖励
        log_probs = []
        rewards = []

        # 生成 episode 数据，pi(theta) 是一个概率分布，我们根据它来采样动作
        for t in range(num_steps):
            state_tensor = torch.FloatTensor(state).unsqueeze(0)  # 转换成形状为 (1, state_dim) 的张量

            # 神经网络输出动作概率分布
            action_probs = policy_net(state_tensor) # 得到一个形状为 (1, action_dim) 的张量，表示在当前状态下每个动作的概率

            # 根据概率分布进行抽样
            m = Categorical(action_probs) # 根据传入的概率张量 action_probs, 构建一个分类分布 Categorical Distribution
            action = m.sample() # 从这个分布中抽样一个动作，得到一个形状为 (1,) 的张量，里面是一个整数，表示抽到的动作索引

            # 记录当时抽取中这个动作的 log 概率，后面更新时会用到 ln pi(a_t|s_t, theta_t)
            log_probs.append(m.log_prob(action)) # m.log_prob(action) 返回一个形状为 (1,) 的张量，表示在当前状态下抽到这个动作的 log 概率

            # 执行动作，得到下一个状态、奖励和是否结束
            next_state, reward, done = env.step(action.item())
            rewards.append(reward)

            if done:
                break
            state = next_state
        
        episode_total_reward = sum(rewards)
        return_history.append(episode_total_reward)

        # Value update 计算 q_t(s_t, a_t)
        # 由于是 Monte Carlo 方法，我们直接用从 t 时刻开始的累计奖励来估计 q_t(s_t, a_t)
        returns = []
        G = 0
        for r in reversed(rewards): # 从后往前计算累计奖励
            G = r + GAMMA*G
            returns.insert(0, G) # 把当前的 G 插入到 returns 的开头，这样最终 returns[i] 就是从 t=i 时刻开始的累计奖励
        
        returns = torch.FloatTensor(returns) # 转换成形状为 (step_t,) 的张量, step_t为这个 episode 的实际步数
        
        # 将回报标准化，能极大地稳定梯度
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        # Policy update 目标是最大化 J(theta) = E[ sum_t ln pi(a_t|s_t, theta) * q_t(s_t, a_t) ]
        policy_loss = []
        for log_prob, G in zip(log_probs, returns): # 广播机制，log_probs 和 returns 的长度都是 step_t
            policy_loss.append(-log_prob * G) # 注意这里是负号，因为我们要最大化 J(theta)，等价于最小化 -J(theta), 形状为 step_t 的list

        # 把所有时间步的损失加起来，得到这个 episode 的总损失
        optimizer.zero_grad()
        loss = torch.cat(policy_loss).sum() # 先把 policy_loss 中的所有张量连接成一个形状为 (step_t,) 的张量，然后求和得到一个形状为 () 的标量张量，表示这个 episode 的总损失

        # 只要0维标量的计算中有神经网络参与，PyTorch 就会自动构建计算图，只需要调用 backward() 来计算梯度
        loss.backward()
        optimizer.step()

        if (ep+1)%100 == 0:
            print(f"Episode {ep+1}/{num_episodes}, Total Reward: {episode_total_reward}")
    
    return policy_net, return_history


In [ ]:
env = GridEnv(GRID_SIZE, GOAL_STATE, OBSTACLE, ACTION, obstacle_through)

trained_policy_net, return_history = train_reinforce(
    env, num_episodes, num_steps, 
    state_dim, action_dim, 
    GAMMA, ALPHA, 
    obstacle_through
    )
plot_learning_curve(return_history)

Episode 100/2000, Total Reward: -590.5000000000068
Episode 200/2000, Total Reward: -105.39999999999824


In [ ]:
Q_final_table = extract_q_table_from_net(trained_policy_net, env, GRID_SIZE, action_dim)
print("Extracted Q-table shape:", Q_final_table.shape)
final_policy, final_V = extract_policy(GRID_SIZE, OBSTACLE, Q_final_table, obstacle_through)
visualize_grid(final_V, final_policy, ALPHA, OBSTACLE, obstacle_through)